# rebound_representation_separability_v1

Purpose: diagnose whether current SOXL/SOXX/router features make rebound states cross-fold separable before another PPO or loss-weighting run.

Why now:
- `state_objective_rebound_target_enrichment_v1` showed target enrichment did not improve original rebound fit.
- `state_objective_state_conditioned_loss_v1` showed BC loss can memorize full-fit rebound rows (`1.000`) but fold-heldout rebound accuracy remains `0.000`, a direct overfit signal.
- This notebook tests feature representation groups with fold-heldout centroid/AUC diagnostics.

External references used:
- Qlib research harness: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- FinRL train/validate/test discipline: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- Supervised contrastive / representation separability idea: https://arxiv.org/abs/2004.11362
- Contrastive predictive coding representation learning: https://arxiv.org/abs/1807.03748
- Class imbalance context from prior failed loss screen: https://arxiv.org/abs/1901.05555

Governance:
- Baseline: `state_objective_state_conditioned_loss_v1`.
- Role: diagnose.
- Changed layer: feature/representation diagnostics, not training.
- Target window: fold-heldout `wf_2022_train` and `wf_2023_2024_train` target rows.
- Acceptance: all folds must have rebound coverage and best feature set must clear holdout AUC/accuracy gates before a state-head/PPO handoff.


In [1]:

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from src.soxl_quant.app.representation_diagnostics import score_binary_representation_feature_sets

EXPERIMENT = 'rebound_representation_separability_v1'
OUT_DIR = Path('runtime/model_selection/card') / EXPERIMENT
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGETS_PATH = Path('runtime/model_selection/card/state_objective_fold_native_teacher_router_v1/fold_native_teacher_targets.csv')
BASELINE_LOSS_SELECTION = Path('runtime/model_selection/card/state_objective_state_conditioned_loss_v1/selection.json')
BASELINE_ENRICHMENT_SELECTION = Path('runtime/model_selection/card/state_objective_rebound_target_enrichment_v1/selection.json')

with BASELINE_LOSS_SELECTION.open() as fh:
    baseline_loss = json.load(fh)
with BASELINE_ENRICHMENT_SELECTION.open() as fh:
    baseline_enrichment = json.load(fh)

targets = pd.read_csv(TARGETS_PATH)
targets['datetime'] = pd.to_datetime(targets['datetime'])
targets['is_rebound_cluster'] = targets['is_rebound_cluster'].fillna(False).astype(bool)
targets['fold'] = targets['fold'].astype(str)

base_context = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'source_targets': str(TARGETS_PATH),
    'rows': int(len(targets)),
    'fold_counts': {str(k): int(v) for k, v in targets['fold'].value_counts().to_dict().items()},
    'rebound_by_fold': {str(k): int(v) for k, v in targets.groupby('fold')['is_rebound_cluster'].sum().to_dict().items()},
    'baseline_loss_decision': baseline_loss['decision'],
    'baseline_loss_holdout_rebound_min': float(baseline_loss['best_holdout_rebound_accuracy_min']),
    'baseline_enrichment_rebound_accuracy': float(baseline_enrichment['best_original_rebound_accuracy']),
}
print(json.dumps(base_context, indent=2, sort_keys=True))


{
  "baseline_enrichment_rebound_accuracy": 0.42857142857142855,
  "baseline_loss_decision": "state_conditioned_loss_partial_rebound_fit_no_promotion",
  "baseline_loss_holdout_rebound_min": 0.0,
  "created_at": "2026-05-25T13:19:02.922117+00:00",
  "experiment": "rebound_representation_separability_v1",
  "fold_counts": {
    "wf_2022_train": 505,
    "wf_2023_2024_train": 756
  },
  "rebound_by_fold": {
    "wf_2022_train": 4,
    "wf_2023_2024_train": 3
  },
  "rows": 1261,
  "source_targets": "runtime/model_selection/card/state_objective_fold_native_teacher_router_v1/fold_native_teacher_targets.csv"
}


In [2]:

feature_sets = {
    'momentum_drawdown': ['mom_5', 'mom_21', 'drawdown_63'],
    'volatility': ['rv_21', 'vol_of_vol_21'],
    'soxx_relative': ['soxl_soxx_ratio_widen_21', 'soxl_soxx_ratio_z_252', 'soxx_mom_21'],
    'router_positions': ['base_position', 'upside_position', 'router_position'],
    'momentum_soxx_relative': ['mom_5', 'mom_21', 'drawdown_63', 'soxl_soxx_ratio_widen_21', 'soxx_mom_21'],
    'all_candidate_features': [
        'mom_5', 'mom_21', 'drawdown_63', 'rv_21', 'vol_of_vol_21',
        'soxl_soxx_ratio_widen_21', 'soxl_soxx_ratio_z_252', 'soxx_mom_21',
        'base_position', 'upside_position', 'router_position',
    ],
}

result = score_binary_representation_feature_sets(
    targets,
    label_col='is_rebound_cluster',
    fold_col='fold',
    feature_sets=feature_sets,
    min_positive_train=2,
    min_positive_eval=2,
    min_holdout_auc=0.70,
    min_holdout_accuracy=0.60,
)

feature_set_scores = result.feature_set_scores
fold_metrics = result.fold_metrics
feature_effects = result.feature_effects

feature_set_scores.to_csv(OUT_DIR / 'feature_set_scores.csv', index=False)
fold_metrics.to_csv(OUT_DIR / 'fold_metrics.csv', index=False)
feature_effects.to_csv(OUT_DIR / 'feature_effects.csv', index=False)

print('feature set scores')
display(feature_set_scores)
print('fold metrics')
display(fold_metrics)
print('top effects')
display(feature_effects.assign(abs_effect=lambda x: x['effect_size'].abs()).sort_values('abs_effect', ascending=False).head(20))


feature set scores


,feature_set,features,feature_count,folds,covered_folds,coverage_gate,holdout_auc_min,holdout_auc_mean,holdout_accuracy_min,holdout_accuracy_mean,representation_gate
0,momentum_drawdown,"mom_5,mom_21,drawdown_63",3,2,2,True,0.980080,0.982056,0.788119,0.837842,True
1,momentum_soxx_relative,"mom_5,mom_21,drawdown_63,soxl_soxx_ratio_widen...",5,2,2,True,0.966799,0.978160,0.790099,0.828251,True
2,all_candidate_features,"mom_5,mom_21,drawdown_63,rv_21,vol_of_vol_21,s...",11,2,2,True,0.880921,0.907277,0.829703,0.873185,True
3,soxx_relative,"soxl_soxx_ratio_widen_21,soxl_soxx_ratio_z_252...",3,2,2,True,0.857459,0.864857,0.752475,0.808777,True
4,router_positions,"base_position,upside_position,router_position",3,2,2,True,0.639222,0.655379,0.699010,0.800563,False
5,volatility,"rv_21,vol_of_vol_21",2,2,2,True,0.037924,0.102185,0.432540,0.618250,False


fold metrics


,feature_set,eval_fold,features,train_samples,eval_samples,train_positive,eval_positive,eval_negative,coverage_gate,holdout_auc,holdout_accuracy
0,all_candidate_features,wf_2022_train,"mom_5,mom_21,drawdown_63,rv_21,vol_of_vol_21,s...",756,505,3,4,501,True,0.933633,0.829703
1,all_candidate_features,wf_2023_2024_train,"mom_5,mom_21,drawdown_63,rv_21,vol_of_vol_21,s...",505,756,4,3,753,True,0.880921,0.916667
2,momentum_drawdown,wf_2022_train,"mom_5,mom_21,drawdown_63",756,505,3,4,501,True,0.984032,0.788119
3,momentum_drawdown,wf_2023_2024_train,"mom_5,mom_21,drawdown_63",505,756,4,3,753,True,0.980080,0.887566
4,momentum_soxx_relative,wf_2022_train,"mom_5,mom_21,drawdown_63,soxl_soxx_ratio_widen...",756,505,3,4,501,True,0.989521,0.790099
5,momentum_soxx_relative,wf_2023_2024_train,"mom_5,mom_21,drawdown_63,soxl_soxx_ratio_widen...",505,756,4,3,753,True,0.966799,0.866402
6,router_positions,wf_2022_train,"base_position,upside_position,router_position",756,505,3,4,501,True,0.639222,0.699010
7,router_positions,wf_2023_2024_train,"base_position,upside_position,router_position",505,756,4,3,753,True,0.671536,0.902116
8,soxx_relative,wf_2022_train,"soxl_soxx_ratio_widen_21,soxl_soxx_ratio_z_252...",756,505,3,4,501,True,0.872255,0.752475
9,soxx_relative,wf_2023_2024_train,"soxl_soxx_ratio_widen_21,soxl_soxx_ratio_z_252...",505,756,4,3,753,True,0.857459,0.865079


top effects


,feature_set,feature,positive_mean,negative_mean,effect_size,positive_count,negative_count,abs_effect
15,momentum_soxx_relative,mom_21,0.503226,0.051860,1.642033,7,1254,1.642033
2,all_candidate_features,mom_21,0.503226,0.051860,1.642033,7,1254,1.642033
12,momentum_drawdown,mom_21,0.503226,0.051860,1.642033,7,1254,1.642033
24,soxx_relative,soxx_mom_21,0.160597,0.021887,1.525160,7,1254,1.525160
8,all_candidate_features,soxx_mom_21,0.160597,0.021887,1.525160,7,1254,1.525160
18,momentum_soxx_relative,soxx_mom_21,0.160597,0.021887,1.525160,7,1254,1.525160
6,all_candidate_features,soxl_soxx_ratio_widen_21,0.294465,0.013118,1.487082,7,1254,1.487082
22,soxx_relative,soxl_soxx_ratio_widen_21,0.294465,0.013118,1.487082,7,1254,1.487082
17,momentum_soxx_relative,soxl_soxx_ratio_widen_21,0.294465,0.013118,1.487082,7,1254,1.487082
13,momentum_drawdown,mom_5,0.223600,0.012285,1.471694,7,1254,1.471694


In [3]:

best = feature_set_scores.iloc[0].to_dict() if not feature_set_scores.empty else {}
representation_gate = bool(best.get('representation_gate', False))
coverage_gate = bool(best.get('coverage_gate', False))

if representation_gate:
    decision = 'rebound_representation_crossfold_separable_no_runtime_promotion'
    next_mechanism = 'state_head_or_ppo_recipe_with_best_rebound_feature_set'
    blocker = 'Representation gate passed, but this is a diagnostic screen and still requires walk-forward PPO/fixed-split validation.'
elif coverage_gate:
    decision = 'rebound_representation_coverage_ok_but_not_separable_no_promotion'
    next_mechanism = 'engineer_rebound_state_head_or_new_features_before_ppo'
    blocker = 'Rebound labels exist in each fold, but current feature sets do not clear cross-fold separability gates.'
else:
    decision = 'rebound_representation_coverage_failed_no_promotion'
    next_mechanism = 'fold_local_rebound_target_discovery_before_state_head'
    blocker = 'At least one feature set/fold lacks enough rebound positives for anti-overfit evaluation.'

selection = {
    'experiment': EXPERIMENT,
    'created_at': base_context['created_at'],
    'source_experiment': 'state_objective_state_conditioned_loss_v1',
    'direct_comparison_baseline': 'state_objective_rebound_target_enrichment_v1',
    'decision': decision,
    'promotion_allowed': False,
    'full_gate': False,
    'representation_gate': representation_gate,
    'coverage_gate': coverage_gate,
    'promotion_blocker': blocker,
    'best_feature_set': str(best.get('feature_set', '')),
    'best_features': str(best.get('features', '')),
    'best_holdout_auc_min': float(best.get('holdout_auc_min', np.nan)),
    'best_holdout_auc_mean': float(best.get('holdout_auc_mean', np.nan)),
    'best_holdout_accuracy_min': float(best.get('holdout_accuracy_min', np.nan)),
    'best_holdout_accuracy_mean': float(best.get('holdout_accuracy_mean', np.nan)),
    'baseline_loss_holdout_rebound_min': float(base_context['baseline_loss_holdout_rebound_min']),
    'baseline_enrichment_rebound_accuracy': float(base_context['baseline_enrichment_rebound_accuracy']),
    'next_mechanism': next_mechanism,
    'acceptance_gate_for_next_run': {
        'relative_return_min_pp': 30.0,
        'max_dd_max_pct': 15.0,
        'fixed_split_required': True,
        'walkforward_required': True,
        'rebound_feature_holdout_auc_min': 0.70,
        'rebound_feature_holdout_accuracy_min': 0.60,
        'holdout_rebound_accuracy_min': 0.60,
        'no_runtime_promotion_from_representation_screen': True,
    },
    'external_sources': [
        'https://github.com/microsoft/qlib',
        'https://arxiv.org/abs/2009.11189',
        'https://github.com/AI4Finance-Foundation/FinRL',
        'https://arxiv.org/abs/2011.09607',
        'https://arxiv.org/abs/2004.11362',
        'https://arxiv.org/abs/1807.03748',
        'https://arxiv.org/abs/1901.05555',
    ],
    'sidecars': [
        'base_context.json',
        'feature_set_scores.csv',
        'fold_metrics.csv',
        'feature_effects.csv',
        'summary.json',
    ],
}

summary = {
    'base_context': base_context,
    'best_row': best,
    'selection': selection,
    'conclusion': [
        f"Best feature set {selection['best_feature_set']} had holdout AUC min/mean {selection['best_holdout_auc_min']:.3f}/{selection['best_holdout_auc_mean']:.3f}.",
        f"Best holdout accuracy min/mean was {selection['best_holdout_accuracy_min']:.3f}/{selection['best_holdout_accuracy_mean']:.3f}; representation gate={selection['representation_gate']}.",
        f"Decision: {selection['decision']}; next mechanism: {selection['next_mechanism']}.",
        "No runtime or PPO promotion is allowed from this representation-only screen.",
    ],
}

(OUT_DIR / 'base_context.json').write_text(json.dumps(base_context, indent=2, sort_keys=True, default=str) + '\n')
(OUT_DIR / 'selection.json').write_text(json.dumps(selection, indent=2, sort_keys=True, default=str) + '\n')
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True, default=str) + '\n')

print(json.dumps(selection, indent=2, sort_keys=True))


{
  "acceptance_gate_for_next_run": {
    "fixed_split_required": true,
    "holdout_rebound_accuracy_min": 0.6,
    "max_dd_max_pct": 15.0,
    "no_runtime_promotion_from_representation_screen": true,
    "rebound_feature_holdout_accuracy_min": 0.6,
    "rebound_feature_holdout_auc_min": 0.7,
    "relative_return_min_pp": 30.0,
    "walkforward_required": true
  },
  "baseline_enrichment_rebound_accuracy": 0.42857142857142855,
  "baseline_loss_holdout_rebound_min": 0.0,
  "best_feature_set": "momentum_drawdown",
  "best_features": "mom_5,mom_21,drawdown_63",
  "best_holdout_accuracy_mean": 0.8378424747236628,
  "best_holdout_accuracy_min": 0.7881188118811882,
  "best_holdout_auc_mean": 0.9820558087013225,
  "best_holdout_auc_min": 0.9800796812749004,
  "coverage_gate": true,
  "created_at": "2026-05-25T13:19:02.922117+00:00",
  "decision": "rebound_representation_crossfold_separable_no_runtime_promotion",
  "direct_comparison_baseline": "state_objective_rebound_target_enrichment_v1",


## Conclusion

The executed notebook writes the final single-run conclusion to `selection.json` and `summary.json`. The key rule is that a representation screen can only permit a future training recipe; it cannot promote a model or actor by itself.
